# NCHU Rental Model — Colab T4 Training

**使用前請確認**：Runtime → Change runtime type → **T4 GPU**

流程：
1. 確認 GPU
2. Clone repo
3. 安裝依賴（含 torch 2.0.1，確保 TorchScript ONNX export）
4. 設定超參數
5. 確認訓練資料
6. 訓練模型（約 27 分鐘）
7. Export + 量化 ONNX
8. 下載量化 ONNX 模型

## Cell 0 — 確認 GPU

In [10]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
else:
    raise RuntimeError('⚠️  沒有 GPU！請到 Runtime → Change runtime type → T4 GPU 再重跑')

CUDA available: True
GPU: Tesla T4
VRAM: 15.6 GB


## Cell 1 — Clone Repo

In [11]:
import os

REPO_URL = "https://github.com/eric20041027/Renting-recommendation-ONNX.git"
REPO_DIR = "/content/Renting-recommendation-ONNX"

if os.path.exists(REPO_DIR):
    print('Repo 已存在，更新至最新...')
    %cd {REPO_DIR}
    !git pull
else:
    !git clone {REPO_URL} {REPO_DIR}
    %cd {REPO_DIR}

!git log --oneline -3

Repo 已存在，更新至最新...
/content/Renting-recommendation-ONNX
Already up to date.
34f2fa9 (HEAD -> main, origin/main, origin/HEAD) fix: update colab notebook to use torch 2.0.1 for TorchScript ONNX export, add Drive backup cell
257c887 fix: replace optimum export with torch.onnx.export to fix ShapeInferenceError during quantization
00859f4 feat: regenerate training data from Jun 13 crawl (704 properties)


## Cell 2 — 安裝依賴

> ONNX export 由 `pipeline/model_training/exporter.py` 處理，已加 `dynamo=False`
> 強制走 legacy TorchScript tracer，權重會嵌入單一 .onnx 檔（torch 2.x 相容）。
> **不需降級 torch，也不需 Restart runtime。**

In [ ]:
# 安裝依賴（不需降級 torch；exporter.py 已用 dynamo=False 走 legacy tracer）
!pip install -q \
    'transformers>=4.35' \
    datasets \
    'onnx>=1.14' \
    'onnxruntime>=1.17' \
    'pandas>=2.0' \
    'pydantic>=2.0' \
    accelerate

import torch
print(f'✅ 安裝完成  torch={torch.__version__}')
print('   無需 Restart runtime，直接執行 Cell 3')

## Cell 3 — 設定超參數

In [ ]:
import os

os.environ["SAVED_MODEL_DIR"]      = "/content/saved_models/rbt6_finetuned"
os.environ["ONNX_OUTPUT_PATH"]     = "/content/saved_models/my_custom_model.onnx"
os.environ["QUANTIZED_MODEL_PATH"] = "/content/saved_models/my_custom_model_quant.onnx"

os.environ["BATCH_SIZE"]    = "64"
os.environ["NUM_EPOCHS"]    = "10"
os.environ["LEARNING_RATE"] = "2e-5"
os.environ["MAX_LENGTH"]    = "64"
os.environ["WARMUP_STEPS"]  = "500"
os.environ["EARLY_STOPPING_PATIENCE"] = "3"
os.environ["RANDOM_SEED"]   = "42"
os.environ["ONNX_OPSET_VERSION"] = "14"
os.environ["FP16"] = "false"
os.environ["ENABLE_ADVERSARIAL_TRAINING"] = "true"
os.environ["ENABLE_HARD_MINING"]          = "true"
os.environ["ENABLE_QUANTIZATION"]         = "false"  # pipeline 只 export 完整 ONNX，量化由 Cell 6 處理（per_channel INT8，對齊正式環境）

os.makedirs("/content/saved_models", exist_ok=True)

import torch
print(f'✅ 設定完成  torch={torch.__version__}  BATCH={os.environ["BATCH_SIZE"]}  EPOCHS={os.environ["NUM_EPOCHS"]}')

## Cell 4 — 確認訓練資料

In [ ]:
import json, os
from collections import Counter

train_path = 'data/processed/recommendation_train.json'

if not os.path.exists(train_path):
    print('⏳ 訓練資料不存在，重新生成...')
    !python3 pipeline/data_prep/generate_dataset.py
    print('✅ 生成完成')

for split in ['train', 'dev', 'test']:
    with open(f'data/processed/recommendation_{split}.json') as f:
        data = json.load(f)
    pos = sum(1 for s in data if s['label'] == 1)
    neg = len(data) - pos
    ct  = sum(1 for s in data if s.get('conflict_type'))
    print(f'{split:5s}: {len(data):6d} 筆  pos={pos} neg={neg} ratio=1:{neg/pos:.1f}  conflict_type={ct}')

## Cell 5 — 訓練模型

T4 預計時間：**20–30 分鐘**（early stopping 約 epoch 6）

In [ ]:
!python3 pipeline_runner.py --skip-phase 1 --skip-phase 2

## Cell 5b — 備份 PyTorch 模型到 Google Drive（強烈建議）

防止 Runtime 超時重啟後模型消失，先備份到 Drive。

In [ ]:
from google.colab import drive
import shutil, os

drive.mount('/content/drive')

dst = '/content/drive/MyDrive/renting_model'
os.makedirs(dst, exist_ok=True)

saved_dir = os.environ["SAVED_MODEL_DIR"]
if os.path.exists(saved_dir):
    shutil.copytree(saved_dir, os.path.join(dst, 'rbt6_finetuned'), dirs_exist_ok=True)
    print(f'✅ PyTorch 模型已備份 → {dst}/rbt6_finetuned')
else:
    print('❌ 模型不存在，請確認 Cell 5 訓練完成')

## Cell 6 — INT8 量化

Cell 5 的 pipeline 已用 `exporter.py`（`dynamo=False`）export 出完整 ~228 MB ONNX。
本 cell 將其量化成 ~57 MB INT8 per_channel 模型（對齊正式環境）。
若 ONNX 不存在（Runtime 重啟），會自動從 Drive 還原 PyTorch 模型並重新 export。

In [ ]:
import os, shutil
from onnxruntime.quantization import quantize_dynamic, QuantType

onnx_path  = os.environ["ONNX_OUTPUT_PATH"]
quant_path = os.environ["QUANTIZED_MODEL_PATH"]

# pipeline (Cell 5) 已透過 exporter.py (dynamo=False) 產生完整 ONNX。
# 若沒找到，從 Drive 還原 PyTorch 模型並重新 export。
if not os.path.exists(onnx_path):
    saved_dir = os.environ["SAVED_MODEL_DIR"]
    if not os.path.exists(saved_dir):
        drive_src = '/content/drive/MyDrive/renting_model/rbt6_finetuned'
        if os.path.exists(drive_src):
            os.makedirs(os.path.dirname(saved_dir), exist_ok=True)
            shutil.copytree(drive_src, saved_dir)
            print('✅ 從 Drive 還原 PyTorch 模型')
        else:
            raise FileNotFoundError('找不到 ONNX 也找不到 PyTorch 模型，請先執行 Cell 5')

    print('ONNX 不存在，重新 export（dynamo=False）...')
    import torch
    from transformers import BertForSequenceClassification, BertTokenizer
    model = BertForSequenceClassification.from_pretrained(saved_dir).eval().cpu()
    model.config.use_cache = False
    tok = BertTokenizer.from_pretrained(saved_dir)
    d = tok('測試查詢', '測試房源文字描述', return_tensors='pt',
            max_length=64, padding='max_length', truncation=True)
    os.makedirs(os.path.dirname(onnx_path), exist_ok=True)
    with torch.no_grad():
        torch.onnx.export(
            model,
            (d['input_ids'], d['attention_mask'], d['token_type_ids']),
            onnx_path,
            input_names=['input_ids', 'attention_mask', 'token_type_ids'],
            output_names=['logits'],
            dynamic_axes={'input_ids': {0: 'batch_size'}, 'attention_mask': {0: 'batch_size'},
                          'token_type_ids': {0: 'batch_size'}, 'logits': {0: 'batch_size'}},
            opset_version=14,
            do_constant_folding=True,
            export_params=True,
            dynamo=False,   # 關鍵：legacy tracer，權重嵌入單一檔
        )

size_mb = os.path.getsize(onnx_path) / 1024 / 1024
print(f'ONNX size: {size_mb:.1f} MB  (應約 228 MB)')
if size_mb < 100:
    raise RuntimeError(f'ONNX 只有 {size_mb:.1f} MB，權重未嵌入（dynamo=False 未生效）')

# INT8 per_channel 動態量化（對齊正式環境 commit b345a8b）
print('INT8 per_channel 量化中...')
quantize_dynamic(onnx_path, quant_path, weight_type=QuantType.QInt8, per_channel=True)

size_q = os.path.getsize(quant_path) / 1024 / 1024
print(f'量化 ONNX size: {size_q:.1f} MB  (應約 57 MB)')
print('✅ 完成！執行 Cell 7 下載')

## Cell 7 — 下載量化 ONNX 模型

In [ ]:
from google.colab import files
import os

quant_path = os.environ["QUANTIZED_MODEL_PATH"]

if os.path.exists(quant_path):
    size_mb = os.path.getsize(quant_path) / 1024 / 1024
    print(f'⬇️  下載：{quant_path}  ({size_mb:.1f} MB)')
    files.download(quant_path)
else:
    print('❌ 量化模型不存在，請先執行 Cell 6')

## Cell 8 — （可選）同步量化模型到 Google Drive

In [ ]:
from google.colab import drive
import shutil, os

drive.mount('/content/drive')

dst = '/content/drive/MyDrive/renting_model'
os.makedirs(dst, exist_ok=True)

quant_path = os.environ["QUANTIZED_MODEL_PATH"]
if os.path.exists(quant_path):
    shutil.copy(quant_path, dst)
    print(f'✅ 量化模型 → {dst}/my_custom_model_quant.onnx')
else:
    print('❌ 量化模型不存在，請先執行 Cell 6')